# 以 ollama 串接 langchain 執行 llama3-8b
變異方式使用 gemini

In [65]:
# 設定必要的資源包和函式
# %pip install ollama
# %pip install langchain chromadb pypdf fastembed
# %pip install aiofiles
# %pip install --upgrade langchain-together
# %pip install langchain_google_genai

In [66]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np
import json

import ollama
from ollama import generate
from langchain_community.vectorstores import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain.schema.output_parser import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate
from langchain.vectorstores.utils import filter_complex_metadata
from langchain_together import ChatTogether
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

## 用來讀取文字檔，並轉成embedding

In [67]:
from typing import AsyncIterator, Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class CustomDocumentLoader(BaseLoader):
    """An example document loader that reads a file line by line."""

    def __init__(self, file_path: str) -> None:
        """Initialize the loader with a file path.

        Args:
            file_path: The path to the file to load.
        """
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:  # <-- Does not take any arguments
        """A lazy loader that reads a file line by line.

        When you're implementing lazy load methods, you should use a generator
        to yield documents one by one.
        """
        with open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1

    # alazy_load is OPTIONAL.
    # If you leave out the implementation, a default implementation which delegates to lazy_load will be used!
    async def alazy_load(
        self,
    ) -> AsyncIterator[Document]:  # <-- Does not take any arguments
        """An async lazy loader that reads a file line by line."""
        # Requires aiofiles
        # Install with `pip install aiofiles`
        # https://github.com/Tinche/aiofiles
        import aiofiles

        async with aiofiles.open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            async for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1

## 設定模型參數

In [68]:
model = 'taide-8b-alpha' # taide-8b-alpha 8b
# model = 'llama3:70b' # llama3:70b
# model = 'llama3' # llama3 8b
temperature = 1
top_p = 0.9
seed = 0
system_prompt = '''你是一個厲害的股市分析師，當正面時回答#非常好或#好，負面時回答#不好或#非常不好，其他則是#無關'''
chunk_size=128
chunk_overlap=10


# f'''<|begin_of_text|><|start_header_id|>system<|end_header_id|> 

#     {system_prompt}<|eot_id|> 
#     <|start_header_id|>user<|end_header_id|>

#     {user_message}<|eot_id|>
#     <|start_header_id|>assistant<|end_header_id|>'''

In [69]:
class RAG:
    vector_store = None
    retriever = None
    chain = None

    def __init__(self):
        self.model = ChatOllama(model=model, max_tokens=128, temperature=temperature, top_p=top_p)
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.prompt = PromptTemplate.from_template(
            """
            <s> [INST] 你是一個厲害的股市分析師，依照使用者的問題回答#好#不好#無關 [/INST] </s> 
            [INST] 問題: {question} 
            文章: {context} 
            回答: [/INST]
            """
        )

    def ingest(self, pdf_file_path: str):
        docs = CustomDocumentLoader(file_path=pdf_file_path).load()
        chunks = self.text_splitter.split_documents(docs)
        chunks = filter_complex_metadata(chunks)
        # chunks.append(chunks)

        vector_store = Chroma.from_documents(documents=chunks, embedding=FastEmbedEmbeddings())
        self.retriever = vector_store.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={
                "k": 3,
                "score_threshold": 0.5,
            },
        )

        self.chain = ({"context": self.retriever, "question": RunnablePassthrough()}
                      | self.prompt
                      | self.model
                      | StrOutputParser())

    def ask(self, query: str):
        if not self.chain:
            return "Please, add a document first."

        return self.chain.invoke(query)

    def clear(self):
        self.vector_store = None
        self.retriever = None
        self.chain = None


In [70]:
# # 測試是否能夠正常運行
# import os

# rag = RAG()
# rag.clear()
# rag.ingest(os.path.dirname(os.path.abspath(os.getcwd()))  + "/stock_news/3023/3023_news/20050129")

# response = rag.ask("請根據信邦 (3023)的商業創新計畫的全面性分析，重點關注其對目標市場的預期影響，並強調由投資者樂觀的長期成長預期所推動的股價持續飙升現象進行判斷")   # Please, add a PDF document first.

# print(response)

In [71]:
# 轉換回答變成訊號
def response_to_signal(ragfile, instruction):
    '''
        ragfile: string 檔案路徑
        instruction: string 問題指令
    '''
    
    text = ''
    try:
        rag = RAG()
        rag.clear()
        rag.ingest(ragfile)
        text = rag.ask(instruction)
        print(text)
    except Exception as e:
        print(e)
        pass
    # print(text)

    if text.find('非常不好') > 0:
        signal = -2
    elif text.find('非常好') > 0:
        signal = 2
    elif text.find('不好') > 0:
        signal = -1
    elif text.find('好') > 0:
        signal = 1
    else:
        signal = 0

    return signal

In [72]:
# content = '''在半導體供應鏈中，IC測試業雖屬後段服務，卻是不可或缺的一環，隨著全球IDM大廠將測試產能委外的發展趨勢逐漸明顯，測試業技術挑戰性越來越高，發展空間也加大。  1顆IC的形成，包括設計、生產製造、封裝到測試，在半導體供應鏈分工過程中，測試是最後階段的工程服務，但在結合前段設計、生產製造及後段封裝的一貫化服務策略下，測試業逐漸發展成為具競爭力的分工夥伴。研發工程師需求增多研發工程師需求增多測試業做些什麼呢？1位測試廠處長表示，測試主要在檢測IC的功能，包括模擬老化、依良率差別分等級、建立有效資料供分析使用等。因IC製程微縮速度愈加快速，整合及複雜度漸高，測試業除了生產管理、設備、測試、品保、軟硬體等各類別工程師之外，各公司為了提高競爭力，漸漸投入技術研發，對研發工程師需求也增多。請繼續往下閱讀...台灣測試廠至少30家台灣測試廠至少30家台灣測試業至少超過30家，知名上市上櫃公司包括京元電、力成、泰林、矽格、欣銓、久元等，另有在海外掛牌的南茂、聯測等公司。各家公司測試產品別互有差異，有的以記憶體為主，代表性公司如力成、泰林等，京元電的邏輯產品比重較高，欣銓以12吋晶圓測試為主，日月光、矽品則以封裝為主。  業者表示，國際全球IDM（整合元件）大廠為了降低營運成本，近幾年開始，陸續釋出後段封測訂單委外代工，這使得台灣測試業營收明顯上揚，不過，隨著技術提高，測試機每台達數千萬元或上億元，測試業已成為資本密集化，同時也需透過與國際大廠策略聯盟，才有競爭力可言，這顯示測試業將走向「大者恆大」局面。應試基本條件︰電學、程式、英文應試基本條件︰電學、程式、英文起薪行情3萬2到3萬6起薪行情3萬2到3萬6記者洪友芳／專題報導工程師為測試業核心專業人力，需求人才以電子、電機、材料等理工相關科系畢業者為主，大學畢業無經驗的新人，起薪行情約3萬2千元到3萬6千元之間，依科系別及個人潛力而定；研究所畢業起薪標準約在3萬9千元上下。1位測試大廠主管表示，測試業工程師至少需要具備基本電學、軟體程式及英文能力，因設備機台是測試業最大的資本支出，也都以英文顯現，客戶有不少比例為國外廠商，工程師需懂電學、程式及英文，才能勝任測試業工作。此外，這位主管也說，測試業沒有自有產品，屬於專業服務業，以測試客戶產品居多，工程師經常需要跟客戶接洽、溝通，滿足客戶需求，工程師的服務熱忱、待人處事態度也要能屈能伸，公司的在職進修於安排專業課程之外，也會有人際溝通等強化客戶服務課程。下班時間常不固定下班時間常不固定京元電人力資源處處長柯芳蓉指出，測試業所需的專業人力，部分是跟前段的IC設計、生產製造相互競爭，基本上，前段的待遇稍高，工作性質也有別，但有的求職者偏好後段測試業，認為發展空間較大。柯芳蓉表示，測試業的工程師以常態班為主，上班時間通常是上午8點多到5點半或6點，但由於半導體業都屬責任制，測試業工程師的下班時間其實並不固定，有時因應客戶交貨趕時間，延到8、9點或半夜下班也是常有的事。論工作甘苦，辛苦是半導體業的共通處，但為客戶解決問題、令客戶滿意時，卻讓人很有工作成就感。封測類股昨於盤中紛跌停，評價： '''
# lm_response(content)

In [73]:
# 計算隔天當沖準確率（開盤買進收盤賣出or開盤賣出收盤回補）
def eval(signal_l, price):
    '''
        signal: list, [日期, 評分]
        price: dataframe, 股票價格
    '''
    tp = 0
    tn = 0
    fp = 0
    fn = 0
    rtns = []   # 記錄當日沖銷的報酬率

    for idx, item in enumerate(signal_l):
        signal = float(item[1])
        index_location = price.index.get_loc(item[0])  # 獲取特定索引的位置
        if index_location == len(price.index) - 1: # 避免超出索引
            break
        next_index = price.index[index_location + 1]  # 獲取下一個索引的位置
        next_row = price.loc[next_index]  # 找到下一個索引的列
        open = float(next_row['Open'])
        close = float(next_row['Close'])
        
        if signal > 0:
            rtns.append(close/open)
            if close > open:
                tp += 1
            elif close < open:
                fp += 1
            else:   
                pass

        elif signal < 0:
            rtns.append(open/close)
            if close < open:
                tn += 1
            elif close > open:
                fn += 1
            else:   
                pass
    if (tp + tn + fp + fn)  == 0:
        accuracy = 0
    else:
        accuracy = (tp + tn) / (tp + tn + fp + fn) # 準確率
        print(tp, tn, fp, fn)

    if len(rtns) == 0:
        avg_rtn = 0
    else:
        avg_rtn = sum(rtns)/len(rtns)  # 這檔股票在這段時間的平均報酬率
    return accuracy, avg_rtn

In [74]:
def write_output(content, raw_data, folder_path):
    '''
        json_content: string, json格式的資料
        raw_data: list, 評分的資料
        folder_path: string, 資料夾路徑
    '''
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(content)
    
    # for i in range(len(raw_data)):
    #     #寫入
    #     file = folder_path + "/mutate" + str(i+1) + ".csv"
    #     pd.DataFrame(raw_data[i]).to_csv(file)
    print("寫入" + folder_path)

In [75]:
# 提示變異
def mutate_prompt(p_mutate):
    '''
        p_mutate: string, 需要變異的提示
    '''
    llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
    prompt = '產生以下指令的1個變體：'
    
    response = llm.invoke(prompt + p_mutate)
    return response.content

# out = mutate_prompt('對業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢')
# print(out)

# 主程式
針對單一股票變異固定次數，分別對每個股票進行變異

In [81]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']

mutate_count = 1 # 變異次數

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ['2880', '2211', '2609']
stock_names = ['華南金', '長榮鋼', '陽明']

for idx, stock_id in enumerate(stock_ids):
    raw_data = []
    accs = []
    rtns = []
    p_org = []
    p_mut = []
    p_origin = '對業務創新計劃進行詳盡審視，重點關注其對目標受眾的預期影響。深入探討投資方的長期成長預期如何為股價持續上漲奠定基礎，並確定支撐此信念的關鍵敺動力。'
    p_mutate = p_origin

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    for i in range(mutate_count):
        acc_ = 0
        print(f'變異{i + 1}次\t執行{stock_id} {stock_names[idx]}')
        qurey = f'請依據"{p_mutate}"對（{stock_id}）{stock_names[idx]}的影響判斷'
        signal_2 = []
        counter = 0
        for date in df_price.index:
            # 檢查日期是否在參考日期之後
            try:
                date_obj = datetime.strptime(date, '%Y%m%d')
            except:
                continue

            if not (start_time <= date_obj <= end_time):
                continue
            
            sig = 0
            for index, file_name in enumerate(sorted_news_files):
                if file_name == date:
                    # sig = response_to_signal(folder_path + file_name, qurey)    # file_name, qurey
                    # print(f'{date}')
                    counter += 1
                    break
            
            signal_2.append([date, sig]) # [日期, 訊號]
            # print(f'{date}: {sig}')

        print(counter)
        raw_data.append(signal_2)
        acc, rtn = eval(signal_2, df_price) # 得到這檔股票的準確率和報酬率
        accs.append(acc)
        rtns.append(rtn)
        p_org.append(p_origin)
        p_mut.append(p_mutate)

        if acc > acc_:
            p_origin = p_mutate
            acc_ = acc
        # p_mutate = mutate_prompt(p_origin)

    # 輸出結果 json file
    output_data = {
        "mutate_count": mutate_count,
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "system_prompt": system_prompt,
        "prompt_mutate": p_mut,
        "prompt_origin": p_org,
        "acc": accs,
        "rtn": rtns,
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "raw_data": raw_data
    }

    # 轉換成 JSON 格式
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    # write_output(content=json_content, raw_data=raw_data, folder_path="eval_llama/output052601/" + stock_id + "/")

變異1次	執行2880 華南金
143
變異1次	執行2211 長榮鋼
49
變異1次	執行2609 陽明
76


In [ ]:
id = '2211'
with open("eval_llama/output052601 01-23-39-234/" + id + "/detail.json") as f:
    data = json.load(f)
    print(data['raw_data'])
df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + id + "twse.csv", dtype=str)
df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
for d in data['raw_data']:
    print(d)
    print(eval(d, df_price))
    print()

[[['20230103', 0], ['20230104', 0], ['20230105', 0], ['20230106', 0], ['20230109', 0], ['20230110', 2], ['20230111', 0], ['20230112', 0], ['20230113', 0], ['20230116', 0], ['20230117', 0], ['20230130', 0], ['20230131', 0], ['20230201', 0], ['20230202', 0], ['20230203', 0], ['20230206', 0], ['20230207', 0], ['20230208', 0], ['20230209', 0], ['20230210', 2], ['20230213', 0], ['20230214', 0], ['20230215', 2], ['20230216', 1], ['20230217', 1], ['20230220', 0], ['20230221', 0], ['20230222', 0], ['20230223', 0], ['20230224', 0], ['20230301', 0], ['20230302', 0], ['20230303', 1], ['20230306', 0], ['20230307', 0], ['20230308', 0], ['20230309', 0], ['20230310', 1], ['20230313', 1], ['20230314', 2], ['20230315', 2], ['20230316', 0], ['20230317', 0], ['20230320', 1], ['20230321', 0], ['20230322', 0], ['20230323', 0], ['20230324', 0], ['20230327', 0], ['20230328', 0], ['20230329', 0], ['20230330', 0], ['20230331', 0], ['20230406', 0], ['20230407', 0], ['20230410', 2], ['20230411', 0], ['20230412',